# This notebook is used to test other pseudonymization tools

Tools to compare with `mailcom`:
- [Presidio](https://github.com/microsoft/presidio/)
- [Scrubadub](https://github.com/LeapBeyond/scrubadub)

## Install tools if needed

Python 3.10

In [ ]:
%pip install scrubadub

In [ ]:
%pip install scrubadub_spacy scrubadub_stanford

In [ ]:
%pip install "presidio_analyzer[transformers]"
%pip install presidio_anonymizer
# python -m spacy download en_core_web_sm

## Try out the tools

### Presidio

In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import TransformersNlpEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider

In [ ]:
text = """
Dear Dr. Emma Müller,

I hope this email finds you well.

I'm writing to you today from NextAI AG in Munich, Germany. We're keen to discuss the exciting developments from the recent Heidelberg AI Summit 2025. Specifically, we were very interested in the presentation on "Next-Generation Robotics" that took place in Room 306 of the main convention center.

Our team at NextAI AG would love to set up a quick call to discuss potential collaborations following the insights shared. We're thinking of a brief chat around July 15th.

Please let us know if July 15th works for you, or suggest an alternative time.

Best regards,
Max Schneider

--- Forwarded message ---
From: info@heidelbergaisummit.de
Date: Friday, 27 June 2025 at 14:30:06
Subject: Heidelberg AI Summit 2025 - Thank You!

Dear Attendees,

Thank you for making the Heidelberg AI Summit 2025 a resounding success! We truly appreciate your participation and engagement. We look forward to seeing you at future events.

Sincerely, 
The Heidelberg AI Summit Team

"""

In [ ]:
# run with custom model config file

config_content = """
nlp_engine_name: transformers
models:
    - lang_code: en
      model_name:
        spacy: en_core_web_sm
        transformers: xlm-roberta-large-finetuned-conll03-english
ner_model_configuration:
labels_to_ignore:
- O
model_to_presidio_entity_mapping:
    PER: PERSON
    LOC: LOCATION
    ORG: ORGANIZATION
    MISC: MISC
low_confidence_score_multiplier: 0.4
low_score_entity_names: []
"""

# save config to a yaml file
with open("config.yaml", "w") as f:
    f.write(config_content)

# Create NLP engine based on configuration file
provider = NlpEngineProvider(conf_file="config.yaml")
nlp_engine = provider.create_engine()

# Set up the engine, loads the NLP module (spaCy model by default) 
# and other PII recognizers
analyzer = AnalyzerEngine(nlp_engine=nlp_engine)

# Call analyzer to get results
results = analyzer.analyze(text=text, language='en')
print(results)

# Analyzer results are passed to the AnonymizerEngine for anonymization

anonymizer = AnonymizerEngine()

anonymized_text = anonymizer.anonymize(text=text, analyzer_results=results)

print(anonymized_text)

In [ ]:
# run with transformers model name

# Define which transformers model to use
model_config = [{"lang_code": "en", "model_name": {
    "spacy": "en_core_web_sm",  # use a small spaCy model for lemmas, tokens etc.
    "transformers": "xlm-roberta-large-finetuned-conll03-english"
    }
}]

nlp_engine = TransformersNlpEngine(models=model_config)

# Set up the engine, loads the NLP module (spaCy model by default) 
# and other PII recognizers
analyzer = AnalyzerEngine(nlp_engine=nlp_engine)

# Call analyzer to get results
results = analyzer.analyze(text=text, language='en')
print(results)

# Analyzer results are passed to the AnonymizerEngine for anonymization

anonymizer = AnonymizerEngine()

anonymized_text = anonymizer.anonymize(text=text, analyzer_results=results)

print(anonymized_text)

In [ ]:
for item in results:
    print(f"Entity: {item.entity_type}, Start: {item.start}, End: {item.end}, Score: {item.score}")

### Scrubadub

In [ ]:
import scrubadub, scrubadub_spacy, scrubadub_stanford # scrubadub_address requires additional setup, see https://scrubadub.readthedocs.io/en/stable/addresses.html

In [ ]:
text = """
Dear Dr. Emma Müller,

I hope this email finds you well.

I'm writing to you today from NextAI AG in Munich, Germany. We're keen to discuss the exciting developments from the recent Heidelberg AI Summit 2025. Specifically, we were very interested in the presentation on "Next-Generation Robotics" that took place in Room 306 of the main convention center.

Our team at NextAI AG would love to set up a quick call to discuss potential collaborations following the insights shared. We're thinking of a brief chat around July 15th.

Please let us know if July 15th works for you, or suggest an alternative time.

Best regards,
Max Schneider

--- Forwarded message ---
From: info@heidelbergaisummit.de
Date: Friday, 27 June 2025 at 14:30:06
Subject: Heidelberg AI Summit 2025 - Thank You!

Dear Attendees,

Thank you for making the Heidelberg AI Summit 2025 a resounding success! We truly appreciate your participation and engagement. We look forward to seeing you at future events.

Sincerely, 
The Heidelberg AI Summit Team

"""

In [ ]:
# add external detectors
scrubber = scrubadub.Scrubber()

# only use one at a time, otherwise the pseudonymized text will have multiple tags for the same entity
scrubber.add_detector(scrubadub_spacy.detectors.SpacyEntityDetector)
# scrubber.add_detector(scrubadub_spacy.detectors.SpacyNameDetector)

# adding the below detectors make the process non-stop running, don't know why
# scrubber.add_detector(scrubadub_stanford.detectors.StanfordEntityDetector)

In [ ]:
pseudonymized_text = scrubber.clean(text)
print(pseudonymized_text)

## Run Presidio on Hugging Face datasets

In [ ]:
# import if needed
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import TransformersNlpEngine

import pandas as pd
import json
import re

In [ ]:
# run with transformers model name

# Define which transformers model to use
model_config = [{"lang_code": "en", "model_name": {
    "spacy": "en_core_web_md",  # use a medium spaCy model to match setting of mailcom
    "transformers": "xlm-roberta-large-finetuned-conll03-english"
    }
}]

nlp_engine = TransformersNlpEngine(models=model_config)

# Set up the engine, loads the NLP module (spaCy model by default) 
# and other PII recognizers
analyzer = AnalyzerEngine(nlp_engine=nlp_engine)


### Email dataset

We used the same dataset as in `quantitative_eval.ipynb` to compare the results of `mailcom` with Presidio.

Run the Prepare data section in `quantitative_eval.ipynb` to get the `"eval/email_detection_eval.csv"` before running this section.

In [ ]:
# load the dataset
email_df = pd.read_csv("eval/email_detection_eval.csv")
# deserialize JSON back to Python objects for emails column
email_df["emails"] = email_df["emails"].apply(json.loads)
email_df.head()

In [ ]:
# Call analyzer to get results
def analyze_row(row):
    text = row["text"]
    results = analyzer.analyze(text=text, language="en")
    saved_results = []
    for item in results:
        if item.entity_type == "EMAIL_ADDRESS":
            saved_results.append(text[item.start:item.end])

    return saved_results

In [ ]:
# add a new column to the dataframe with the analyzer results
email_df["presidio_emails"] = email_df.apply(analyze_row, axis=1)

In [ ]:
# create expected text and actual text after replacing email by [email]
def replace_emails_with_placeholder(text, emails):
    for email in emails:
        text = text.replace(email, "[email]")
    return text

email_df["expected_text"] = email_df.apply(lambda row: replace_emails_with_placeholder(row["text"], row["emails"]), axis=1)
email_df["presidio_text"] = email_df.apply(lambda row: replace_emails_with_placeholder(row["text"], row["presidio_emails"]), axis=1)

In [ ]:
email_df["exact_match"] = email_df.apply(lambda row: row["expected_text"] == row["presidio_text"], axis=1)

In [ ]:
# calculate tp, fp, fn
def calculate_tp_fp_fn(row):
    pred_count = len(re.findall(r"\[email\]", row["presidio_text"]))
    gold_count = len(re.findall(r"\[email\]", row["expected_text"]))

    true_positives = min(pred_count, gold_count)
    false_positives = max(pred_count - gold_count, 0)
    false_negatives = max(gold_count - pred_count, 0)
    
    return pd.Series({"tp": true_positives, "fp": false_positives, "fn": false_negatives})

metrics_df = email_df.apply(calculate_tp_fp_fn, axis=1)
email_df = pd.concat([email_df, metrics_df], axis=1)

In [ ]:
# save the results to a csv file for further analysis
email_df.to_csv("eval/presidio_email_detection_results.csv", index=False)

In [ ]:
def cal_eval_metrics(df):
    total_tp = df["tp"].sum()
    total_fp = df["fp"].sum()
    total_fn = df["fn"].sum()

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    accuracy = df["exact_match"].mean()
    print(f"Exact match accuracy: {accuracy:.4f}")
    print(f"Micro Precision: {precision:.4f}")
    print(f"Micro Recall: {recall:.4f}")
    print(f"Micro F1 Score: {f1:.4f}")


In [ ]:
cal_eval_metrics(email_df)
# Exact match accuracy: 0.9995
# Micro Precision: 1.0000
# Micro Recall: 0.9995
# Micro F1 Score: 0.9998
# There is only one failed case:
# -- sentence: With an email address that reflects her professional prowess-bhamini.gulati@shukla.biz-she navigates the digital realm with ease.
# -- expected content: With an email address that reflects her professional prowess-[email]-she navigates the digital realm with ease.
# -- Presidio did not recognize the email.

In [ ]:
email_df[email_df["exact_match"] == False]

### NER dataset

We used the same dataset as in `quantitative_eval.ipynb` to compare the results of `mailcom` with Presidio.

Run the Prepare data section in `quantitative_eval.ipynb` to get the `"eval/ner_detection_eval.csv"` before running this section.

In [ ]:
# load the dataset
ner_df = pd.read_csv("eval/ner_detection_eval.csv")
# deserialize JSON back to Python objects for entities and tokens columns
ner_df["entities"] = ner_df["entities"].apply(json.loads)
ner_df["tokens"] = ner_df["tokens"].apply(json.loads)

In [ ]:
type_mapping = {
    "PERSON": "PER",
    "LOCATION": "LOC",
    "ORGANIZATION": "ORG",
} # Presidio does not consider MISC type

In [ ]:
# Call analyzer to get results
def analyze_row(row):
    text = row["sentence"]
    results = analyzer.analyze(text=text, language="en")
    saved_results = []
    for item in results:
        if item.entity_type in type_mapping:
            entity_type = type_mapping[item.entity_type]
        else:
            entity_type = item.entity_type
        saved_results.append({
            "type": entity_type,
            "start": item.start,
            "end": item.end,
            "text": text[item.start:item.end],
        })

    return saved_results

In [ ]:
# add a new column to the dataframe with the analyzer results
ner_df["presidio_results"] = ner_df.apply(analyze_row, axis=1) # 19 minutes 4.2 seconds on laptop

In [ ]:
# calculate tp, fp, fn for each row
def calculate_tp_fp_fn(row):
    gold_data = row["entities"]
    pred_data = row["presidio_results"]

    # normalize data to get tuple of (type, text, start, end)
    norm_gold_data = {
        (e["type"], e["text"], e["start"], e["end"]) for e in gold_data if e["type"] in type_mapping.values()
    }
    norm_pred_data = {
        (e["type"], e["text"], e["start"], e["end"]) for e in pred_data if e["type"] in type_mapping.values()
    }

    tp = len(norm_gold_data & norm_pred_data)
    fp = len(norm_pred_data - norm_gold_data)
    fn = len(norm_gold_data - norm_pred_data)

    return pd.Series({"tp": tp, "fp": fp, "fn": fn})

In [ ]:
def cal_eval_metrics(dataframe):
    # calculate precision, recall, and F1 score for all rows
    tp = fp = fn = 0

    # add tp, fp, fn columns to the dataframe
    metrics_df = dataframe.apply(calculate_tp_fp_fn, axis=1)
    result_df = pd.concat([dataframe, metrics_df], axis=1)

    for _, row in result_df.iterrows():
        tp += row["tp"]
        fp += row["fp"]
        fn += row["fn"]

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")

    return result_df

In [ ]:
result_df = cal_eval_metrics(ner_df)
result_df.head()
# Precision: 0.6230
# Recall: 0.4985
# F1 Score: 0.5538

In [ ]:
# save the results to a csv file for further analysis
result_df.to_csv("eval/presidio_ner_detection_results.csv", index=False)